# Barter: Engine, Strategies, Risk & Statistics (Advanced)

This notebook demonstrates the full engine pipeline using the Python bindings:
configuration, backtesting, custom strategies, risk management, multi-strategy
composition, event callbacks, and trading statistics.

> This is the Python equivalent of the Rust `evcxr` notebook,
> using the same underlying Rust engine via PyO3 bindings.

## Topics Covered
- `SystemConfig` JSON structure and engine lifecycle
- Custom strategy callbacks with `EngineState` snapshots
- Multi-strategy composition with per-strategy position tracking
- Custom risk management with order filtering
- Event callbacks: `on_fill`, `on_position_closed`
- `TradingSummaryGenerator` for performance analysis
- Metrics: Sharpe, Sortino, Calmar, max drawdown


## Prerequisites

```bash
cd /path/to/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```


In [1]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


Using barter from /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python/python


In [ ]:
import json
from datetime import datetime, timedelta, timezone

from barter import (
    ExchangeId, Instrument, IndexedInstruments, Balance,
    OrderRequestOpen, OrderRequestCancel, EngineState,
    TradingSummary, TradingSummaryGenerator,
    TradeFill, PositionExited,
    run_backtest,
)



---
## Part 1: Engine & Backtesting


### System Configuration

Barter uses a JSON-based `SystemConfig` that defines:
- **Instruments**: which exchange/asset pairs to trade
- **Executions**: mock or live execution clients with initial balances

```
SystemArgs → SystemBuilder → build() → init() → run → shutdown → TradingSummary
```


In [3]:
# Load example config and market data
config_path = REPO_ROOT / "barter" / "examples" / "config" / "backtest_config.json"
market_data_path = REPO_ROOT / "barter" / "examples" / "data" / "binance_spot_market_data_with_disconnect_events.json"

config = json.loads(config_path.read_text())
system_config = json.dumps(config["system"])
market_data = market_data_path.read_text()

print("Config instruments:", [i["name_exchange"] for i in config["system"]["instruments"]])
print("Config executions:", len(config["system"]["executions"]))


Config instruments: ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
Config executions: 1


In [4]:
# Run baseline backtest (noop strategy)
summary = await run_backtest(system_config, market_data, risk_free_return=0.05)
print(summary)
print(summary.to_json()[:500])


TradingSummary(start='2024-12-20T19:05:18.176373191+00:00', end='2024-12-20T19:05:18.227363589+00:00', duration=0s)
{
  "assets": [
    {
      "asset": "btc",
      "exchange": "binance_spot",
      "summary": {
        "balance_end": {
          "free": "0.1",
          "total": "0.1"
        },
        "drawdown": null,
        "drawdown_max": null,
        "drawdown_mean": null
      }
    },
    {
      "asset": "eth",
      "exchange": "binance_spot",
      "summary": {
        "balance_end": {
          "free": "1",
          "total": "1"
        },
        "drawdown": null,
        "drawdown_max": nul


### Engine Architecture

```
┌─────────────────┐     ┌──────────────────┐
│  Market Data     │────▶│     Engine        │────▶ TradingSummary
│  (WebSocket)     │     │   (Processor)     │
└─────────────────┘     │                    │
                        │  ┌──────────────┐  │
┌─────────────────┐     │  │ EngineState   │  │
│  Account Events  │────▶│  │  - positions  │  │
│  (execution rx)  │     │  │  - balances   │  │
└─────────────────┘     │  │  - orders     │  │
                        │  └──────────────┘  │
┌─────────────────┐     │  ┌──────────────┐  │     ┌──────────────┐
│  Commands        │────▶│  │ Strategy     │──│────▶│  Execution   │
│  (external ctrl) │     │  │ RiskManager  │  │     │  (mock/live) │
└─────────────────┘     │  └──────────────┘  │     └──────────────┘
```


---
## Part 2: Custom Trading Strategies


### Strategy Traits (Rust → Python)

| Rust Trait | Python Equivalent | When Called |
|------------|-------------------|------------|
| `AlgoStrategy` | `strategy=fn(state)` | Every engine tick |
| `ClosePositionsStrategy` | (auto-generated) | On shutdown |
| `OnDisconnectStrategy` | (noop) | On WebSocket disconnect |
| `OnTradingDisabled` | (noop) | When trading state → Disabled |


In [5]:
# Simple strategy: buy BTC every tick
def buy_btc(state):
    price = state.price(0)
    if price:
        return [OrderRequestOpen(0, 0, "buy", price, 0.001)]
    return []

summary = await run_backtest(system_config, market_data, strategy=buy_btc)
print(f"Single strategy: {summary}")


Single strategy: TradingSummary(start='2024-12-20T19:05:18.175078500+00:00', end='2024-12-20T19:05:18.227653983+00:00', duration=0s)


### EngineState Snapshot

The strategy callback receives a read-only snapshot each tick.


In [6]:
# Inspect the state
snapshots = []
def inspect(state):
    if len(snapshots) < 2:
        snapshots.append({
            "instruments": state.instruments(),
            "balances": state.balances(),
            "btc_price": state.price(0),
            "btc_position": state.position(0),
            "open_orders": state.open_orders(0),
        })
    return []

await run_backtest(system_config, market_data, strategy=inspect)
for i, s in enumerate(snapshots):
    print(f"Tick {i}: price={s['btc_price']}, position={s['btc_position']}, orders={s['open_orders']}")


Tick 0: price=None, position=None, orders=0
Tick 1: price=96962.51, position=None, orders=0


---
## Part 3: Multi-Strategy Composition


Pass `strategies={"name": fn, ...}` to run multiple named strategies
in a single engine. Each strategy gets independent position tracking.


In [7]:
def momentum(state):
    price = state.price(0)
    if price:
        return [OrderRequestOpen(0, 0, "buy", price, 0.001, strategy_id="momentum")]
    return []

def mean_reversion(state):
    price = state.price(0)
    if price:
        return [OrderRequestOpen(0, 0, "sell", price, 0.0005, strategy_id="mean_rev")]
    return []

multi = await run_backtest(
    system_config, market_data,
    strategies={"momentum": momentum, "mean_rev": mean_reversion},
)
print(f"Multi-strategy: {multi}")


Multi-strategy: TradingSummary(start='2024-12-20T19:05:18.175031498+00:00', end='2024-12-20T19:05:18.226491988+00:00', duration=0s)


---
## Part 4: Risk Management


### How Risk Fits in the Pipeline

```
Strategy.generate_orders(state)
    │
    ▼
RiskManager.check(state, orders)
    │
    ▼  (only approved orders)
ExecutionClient (submit to exchange)
```


In [8]:
# Risk manager: reject orders with notional > $100
rejected = []

def max_notional_risk(state, orders):
    approved = []
    for o in orders:
        notional = o.price * o.quantity
        if notional <= 100.0:
            approved.append(o)
        else:
            rejected.append(f"Rejected: {o.side} {o.quantity} @ {o.price:.0f} (notional={notional:.0f})")
    return approved

def big_orders(state):
    price = state.price(0)
    if not price: return []
    return [
        OrderRequestOpen(0, 0, "buy", price, 0.001),   # ~$97 → approved
        OrderRequestOpen(0, 0, "buy", price, 0.01),    # ~$970 → rejected
    ]

risk_summary = await run_backtest(system_config, market_data, strategy=big_orders, risk=max_notional_risk)
print(f"Rejected: {len(rejected)} orders")
for r in rejected[:3]:
    print(f"  {r}")


Rejected: 24 orders
  Rejected: buy 0.01 @ 96963 (notional=970)
  Rejected: buy 0.01 @ 96963 (notional=970)
  Rejected: buy 0.01 @ 96963 (notional=970)


### Recommended Risk Checks for Production

1. **Max notional per order** — prevents fat-finger errors
2. **Max notional per instrument** — limits exposure per market
3. **Max total portfolio exposure** — global position limit
4. **Market order price deviation** — prevents slippage on stale prices
5. **Order rate limiting** — prevents runaway strategy loops
6. **Instrument kind whitelist** — only trade supported derivatives
7. **Balance sufficiency** — check free balance before opening


---
## Part 5: Event Callbacks


In [9]:
fills = []
closures = []

def on_fill(trade):
    fills.append(f"{trade.side} {trade.quantity} @ {trade.price:.2f} (strategy={trade.strategy_id})")

def on_position_closed(pos):
    closures.append(f"instrument={pos.instrument_index} PnL={pos.pnl_realised:.2f}")

await run_backtest(
    system_config, market_data,
    strategy=buy_btc,
    on_fill=on_fill,
    on_position_closed=on_position_closed,
)

print(f"Fills: {len(fills)}, Closures: {len(closures)}")
for f in fills[:3]:
    print(f"  {f}")


Fills: 0, Closures: 0


---
## Part 6: Trading Statistics & Performance Analysis


### TradingSummaryGenerator

Build summaries from synthetic balance and position data without running the full engine.


In [10]:
btc = Instrument.spot(ExchangeId.BINANCE_SPOT, "btc_usdt", "BTCUSDT", "btc", "usdt")
eth = Instrument.spot(ExchangeId.BINANCE_SPOT, "eth_usdt", "ETHUSDT", "eth", "usdt")
instruments = IndexedInstruments([btc, eth])

gen = TradingSummaryGenerator(instruments, 0.05)

start = datetime(2025, 1, 1, tzinfo=timezone.utc)
asset_idx = 2  # usdt

# Feed balance updates
for day, total in enumerate([10000, 10400, 10150, 10900]):
    gen.update_balance(asset_idx, total, total, (start + timedelta(days=day)).isoformat().replace("+00:00", "Z"))

# Feed closed positions
positions = [
    (0, "buy", 400, 0.05, 0, 2),
    (0, "buy", -250, 0.04, 3, 5),
    (1, "buy", 650, 0.75, 6, 8),
]
for inst, side, pnl, qty, enter_day, exit_day in positions:
    gen.update_position(inst, side, pnl, qty,
        (start + timedelta(days=enter_day)).isoformat().replace("+00:00", "Z"),
        (start + timedelta(days=exit_day)).isoformat().replace("+00:00", "Z"),
    )

print(gen)


TradingSummaryGenerator(instruments=2, assets=3)


In [11]:
# Generate summary with crypto-style annualization (365 days)
summary = gen.generate("annual365")
print(summary)
print(json.dumps(summary.to_dict(), indent=2)[:2000])


TradingSummary(start='2026-04-10T23:42:24.043126877+00:00', end='2026-04-10T23:42:24.043126877+00:00', duration=0s)
{
  "assets": [
    {
      "asset": "btc",
      "exchange": "binance_spot",
      "summary": {
        "balance_end": null,
        "drawdown": null,
        "drawdown_max": null,
        "drawdown_mean": null
      }
    },
    {
      "asset": "eth",
      "exchange": "binance_spot",
      "summary": {
        "balance_end": null,
        "drawdown": null,
        "drawdown_max": null,
        "drawdown_mean": null
      }
    },
    {
      "asset": "usdt",
      "exchange": "binance_spot",
      "summary": {
        "balance_end": {
          "free": "10900",
          "total": "10900"
        },
        "drawdown": null,
        "drawdown_max": {
          "time_end": "2025-01-04T00:00:00Z",
          "time_start": "2025-01-02T00:00:00Z",
          "value": "0.0240384615384615384615384615"
        },
        "drawdown_mean": {
          "mean_drawdown": "0.02403846

### Available Metrics

| Metric | Description |
|--------|-------------|
| **Total PnL** | Net realised profit/loss |
| **Win Rate** | Percentage of profitable trades |
| **Sharpe Ratio** | Risk-adjusted return (vs risk-free rate) |
| **Sortino Ratio** | Downside-risk-adjusted return |
| **Calmar Ratio** | Return / max drawdown |
| **Max Drawdown** | Largest peak-to-trough decline |
| **Rate of Return** | Annualised return percentage |
| **Profit Factor** | Gross profit / gross loss |

### Time Intervals
- `annual365` — 24/7 crypto markets (365 days/year)
- `daily` — daily statistics without annualisation
